# Blocking-completion index: baseline scan vs incremental index

Compares the historical full-scan `Solver::decide` blocking path (**baseline**)
against the incremental `BlockingCompletionIndex` (**index**) on a universal
corpus of 1,000 conda-forge problems (seed 0, 60 s timeout, release builds).

CSVs are produced by `tools/solve-snapshot --mode universal`; problems are
joined on the unique `index` column. Figures are written next to this notebook
as PNGs.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 150

BENCH = Path("/workspace/bench")
OUT = Path("/workspace/bench/figures")
OUT.mkdir(exist_ok=True)

base_path = BENCH / "uni1000-base.csv"
tip_path  = BENCH / "uni1000-index.csv"
base_label = "baseline (full scan)"
tip_label  = "incremental index"

paths  = [base_path, tip_path]
labels = [base_label, tip_label]

# Join key is the unique problem `index`; `requirements` may contain newlines.
dfs = [pl.read_csv(p).select(pl.col("index"), pl.col("duration"), pl.col("outcome")) for p in paths]
for label, p, df in zip(labels, paths, dfs):
    print(f"{label}: {df.height} problems, {df.select((pl.col('outcome')=='ok').sum()).item()} ok "
          f"({df.select(pl.col('duration').sum()).item():.1f}s total)")

In [ ]:
# --- Figure 1: solve-duration histograms, baseline vs index ---
threshold = 60
bins = [0, 0.25, 0.5, 0.75, 1, 1.5, 2, 3, 4, 5, 7.5, 10, 15, 20, 30, 45, threshold, threshold + 1]

fig, axs = plt.subplots(2, sharex=True, figsize=(9, 5))
stats = []
for label, df, ax in zip(labels, dfs, axs):
    values = df["duration"].to_numpy()
    values = np.clip(values, None, threshold)
    s = dict(label=label, mean=values.mean(), median=np.median(values),
             std=values.std(), p25=np.percentile(values,25), p75=np.percentile(values,75),
             mx=values.max(), timeouts=int((values>=threshold).sum()))
    stats.append(s)
    print(f"{label}: mean {s['mean']:.3f}s  median {s['median']:.3f}s  "
          f"p75 {s['p75']:.3f}s  max {s['mx']:.3f}s  timeouts {s['timeouts']}")
    counts, _, bars = ax.hist(values, bins=bins, density=True)
    ax.set_title(label)
    ax.bar_label(bars, fontsize=7, labels=[f'{b.get_height():.1%}' for b in bars])
    ax.tick_params(axis='y', left=False, labelleft=False)

print(f"\nmean ratio (base/index): {stats[0]['mean']/stats[1]['mean']:.3f}x   "
      f"median ratio: {stats[0]['median']/stats[1]['median']:.3f}x")
fig.supxlabel("Solve duration (seconds)")
fig.supylabel("Share of solves")
fig.suptitle(f"Universal solve-duration histogram (n={dfs[0].height}): {base_label} vs {tip_label}")
plt.tight_layout()
fig.savefig(OUT / "fig1_duration_histogram.png", bbox_inches="tight")
plt.show()

In [ ]:
# --- Figure 2: per-problem difference (index - baseline); negative = index faster ---
joined = dfs[1].join(dfs[0], on="index", suffix="_base")
diff = (joined["duration"] - joined["duration_base"]).to_numpy()
mean_d, med_d = diff.mean(), np.median(diff)

fig, ax = plt.subplots(figsize=(9, 5))
lim = np.percentile(np.abs(diff), 99)  # clip axis to 99th pct so the bulk is visible
b = np.linspace(-lim, lim, 41)
n, b, patches = ax.hist(np.clip(diff, -lim, lim), bins=b, density=True, alpha=0.8)
for bar, left in zip(patches, b[:-1]):
    bar.set_facecolor('#2a9d4a' if left < 0 else '#c0392b')
ax.axvline(mean_d, color='blue', ls='--', lw=1.5, label=f"mean {mean_d*1000:+.1f} ms")
ax.axvline(med_d, color='purple', ls='--', lw=1.5, label=f"median {med_d*1000:+.1f} ms")
ax.axvline(0, color='black', lw=0.8)
ax.set_title(f"Per-problem duration difference ({tip_label} − {base_label})")
ax.set_xlabel("Difference (seconds) — negative = index faster")
ax.set_ylabel("Density")
ax.legend()
ax.grid(axis='y', ls='--', alpha=0.5)
print(f"per-problem diff: mean {mean_d*1000:+.2f} ms, median {med_d*1000:+.2f} ms, "
      f"std {diff.std()*1000:.1f} ms   (faster: {(diff<0).sum()}, slower: {(diff>0).sum()})")
plt.tight_layout()
fig.savefig(OUT / "fig2_perproblem_diff.png", bbox_inches="tight")
plt.show()

In [ ]:
# --- Figure 3: blocking-completion work removed (scan literal-visits vs index recompute + routing) ---
# Parsed from the run logs' per-problem diagnostics lines.
import re
def parse_work(logpath):
    q=cv=lv=routed=occ=0
    for line in open(logpath, errors='replace'):
        if m:=re.search(r'blocking completion: (\d+) queries, (\d+) clause visits, (\d+) literal visits', line):
            q+=int(m[1]); cv+=int(m[2]); lv+=int(m[3])
        if m:=re.search(r'blocking index: \d+ registered, (\d+) routed, (\d+) occurrence visits', line):
            routed+=int(m[1]); occ+=int(m[2])
    return dict(queries=q, clause_visits=cv, literal_visits=lv, routed=routed, occ=occ)

wb = parse_work(BENCH / "uni1000-base.log")
wi = parse_work(BENCH / "uni1000-index.log")
print("baseline scan work:", wb)
print("index work:        ", wi)

fig, ax = plt.subplots(figsize=(8, 4.5))
cats = ["completion\nqueries", "clause\nvisits", "literal\nvisits"]
xb = [wb['queries'], wb['clause_visits'], wb['literal_visits']]
xi = [wi['queries'], wi['clause_visits'], wi['literal_visits']]
x = np.arange(len(cats)); w = 0.38
ax.bar(x - w/2, xb, w, label=base_label, color='#c0392b')
ax.bar(x + w/2, xi, w, label=tip_label, color='#2a9d4a')
for i,(a,b_) in enumerate(zip(xb,xi)):
    ax.text(i-w/2, a, f"{a:,}", ha='center', va='bottom', fontsize=8)
    ax.text(i+w/2, b_, f"{b_:,}", ha='center', va='bottom', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(cats)
ax.set_ylabel("count over 1,000 problems")
ax.set_title("Blocking-completion work: scan vs index\n"
             f"(index also routes {wi['routed']:,} trail vars / visits {wi['occ']:,} occurrences to stay in sync)")
ax.legend()
plt.tight_layout()
fig.savefig(OUT / "fig3_work_counters.png", bbox_inches="tight")
plt.show()